- 显存容量 (GB)
- 算力 (TFLOPS)
    - K => M => G => T
    - tensor cores（$D = A \times B + C$） 的数量和代数；
        - sparse 矩阵加速
    - TFLOPS: Floating point operations per second，是一个速度概念，
        - TFLOPs：复数概念
- 显存带宽 (GB/s) : 显存每秒能把多少数据搬运给核心
    - HBM: high band-width memory
    - 显存带宽是 GPU 芯片（计算核心，SM 流式多处理器）与 显存（VRAM）之间交换数据的速度。
        - $\text{带宽} = \text{显存频率} \times \text{显存位宽} / 8$ （车速 \* 车道数）
            - $\text{Bandwidth (GB/s)} = \frac{\text{Effective Clock (Gbps)} \times \text{Bus Width (bit)}}{8}$
            - Gbps (有效速率) = MHz (基准频率, base clock) × 倍率。
        - 显存带宽指 GPU 在一秒钟内最多能读取或写入多少数据量。GB/s (Gigabytes per second)。
        - SRAM/寄存器：极快，但空间极小（只有几十 MB），根本装不下大模型。
            - 一个 7B 的大模型（FP16精度）需要 14GB 显存。SRAM 根本装不下，所以 GPU 必须不断从 VRAM 把权重搬运到计算核心（SM），这就导致了显存带宽（Memory Bandwidth） 往往比 计算能力（TFLOPS） 更容易成为瓶颈（即 Memory Wall 问题）。尤其是 llm inference 的时候
- nvlink 卡间通信带宽
- 精度支持
    - 架构直接相关
- 参考：
    - https://my.feishu.cn/wiki/WgaPwKSczi8P6jkRmBncEXL7nBf?from=from_copylink
    - https://www.autodl.com/docs/gpu/
    - https://developer.nvidia.com/cuda/gpus
    - https://llm.0.af/

In [8]:
# 9.0: Hopper (如 H100)
# 8.9: Ada Lovelace (如 RTX 4090)
# 8.0 / 8.6: Ampere (如 A100, RTX 3090)
import torch
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
# https://developer.nvidia.com/cuda/gpus

(8, 9)
['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


| 特性参数 | **RTX 4090** <br>(Consumer Flagship) | **RTX 5090** <br>(**New** Consumer Flagship) | **NVIDIA A100** <br>(SXM 80GB) | **NVIDIA A800** <br>(China Export Version) | **NVIDIA H100** <br>(SXM Version) | **NVIDIA H200** <br>(**The New King**) | **Apple M4 Max** <br>(Unified Memory) | **核心差异 (针对 RL & LLM 研究)** |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **显存带宽** | **1,008 GB/s** | **1,792 GB/s** | **2,039 GB/s** | **2,039 GB/s** | **3,350 GB/s** | **4,800 GB/s** | **546 GB/s** | **最核心差异**。H100 带宽是 4090 的 3.3 倍，直接决定 LLM 推理速度上限。<br>**[5090]** 带宽暴涨 78%，接近 A100 水平，推理速度将有质变。<br>**[H200]** 搭载 HBM3e，带宽提升 40% 达到 4.8TB/s，是目前业界最高吞吐，彻底释放 LLM 生成速度。<br>**[A800]** 显存带宽未被阉割，与 A100 一致，单卡推理性能基本无损。<br>**[M4 Max]** 虽不及独立显卡，但在移动端/SoC 中极强，带宽约为 4090 的一半，优于绝大多数消费级 CPU 方案。 |
| **显存容量** | **24 GB** | **32 GB** | **80 GB** | **80 GB** | **80 GB** | **141 GB** | **Up to 128 GB** <br>(Unified) | 4090 无法全参数微调 70B 级模型；A100/H100 可容纳大 Batch Size 训练。<br>**[5090]** 32GB 虽仍不够全微调 70B，但可容纳 70B 量化模型及更长 Context。<br>**[H200]** 141GB 是质变，单卡即可运行 **Llama-3-70B (FP16/BF16)** 而无需量化，推理 KV Cache 容量翻倍。<br>**[A800]** 容量与 A100 一致，大模型微调的显存优势依然存在。<br>**[M4 Max]** **最大亮点**。统一内存架构 (UMA) 允许 CPU/GPU 共享高达 128GB 显存，是运行超大模型 (如 Llama-3-120B 量化版) 成本最低的方案，但仅限推理。 |
| **显存类型** | GDDR6X | **GDDR7** | HBM2e | HBM2e | HBM3 | **HBM3e** | LPDDR5X-8533 | HBM (High Bandwidth Memory) 就像把显存堆叠在芯片旁，延迟低、吞吐大。<br>**[5090]** GDDR7 进一步拉近了与 HBM 的延迟和带宽差距。<br>**[H200]** HBM3e 是目前最快显存技术，专为解决"内存墙"问题而生。<br>**[A800]** 与 A100 相同，维持了高带宽低延迟特性。<br>**[M4 Max]** 使用片上封装的高频 LPDDR5X，延迟极低，但物理原理上仍略逊于 HBM 堆叠。 |
| **显存位宽** | 384-bit | **512-bit** | **5,120-bit** | **5,120-bit** | **5,120-bit** | **6,144-bit** | 512-bit | 4090 靠高频 (21Gbps) 跑数据；H100 靠极宽的车道 (5120-bit) 跑数据。<br>**[5090]** 回归 512-bit 宽车道，基础物理通道更宽。<br>**[H200]** 激活了第 6 颗 HBM 堆栈，物理位宽进一步拓宽至 6144-bit。<br>**[A800]** 位宽未阉割，数据吞吐能力保留。<br>**[M4 Max]** 虽为 SoC，但 512-bit 位宽已达到顶级桌面显卡 (如 5090) 的水平，保障了数据吞吐。 |
| **显存频率** | ~1,313 MHz<br>(等效 **21 Gbps**) | ~1,750 MHz<br>(等效 **28 Gbps**) | ~1,512 MHz<br>(等效 **3.2 Gbps**) | ~1,512 MHz<br>(等效 **3.2 Gbps**) | ~1,313 MHz<br>(等效 **5.3 Gbps**) | ~1,313 MHz<br>(等效 **6.3 Gbps**) | 8,533 Mbps | GDDR6X 虽频率极高，但位宽太窄，总吞吐量依然不如 HBM。<br>**[5090]** 依靠 GDDR7 的超高频 (28Gbps+) 再次拔高吞吐上限。<br>**[H200]** 频率并未大幅提升，主要靠位宽和 HBM3e 技术提升效率。<br>**[A800]** 频率与 A100 一致。<br>**[M4 Max]** 依靠极高的 LPDDR 频率弥补带宽，但在"大规模并行读写"上仍不如 HBM 架构。 |
| **CUDA Cores** | 16,384 | **~21,760** | 6,912 | 6,912 | 16,896 | 16,896 | **N/A** <br>(40 GPU Cores) | 4090 的 FP32 通用计算极强，适合 RL 环境仿真 (Isaac Gym) 等计算密集型任务。<br>**[5090]** 核心数激增 ~33%，单卡仿真效率进一步碾压计算卡。<br>**[H200]** 计算核心与 H100 完全相同 (Hopper 架构)，算力无本质区别。<br>**[A800]** 核心数未变，单卡计算能力 (FP32) 与 A100 相同。<br>**[M4 Max]** 无 CUDA 核心，主要依靠 Metal/MPS 加速。RL 仿真 (Isaac Gym/Sim) 兼容性较差，不适合需要大量 CUDA 交互的训练任务。 |
| **Tensor Cores** | 512 (第 4 代) | **680 第 5 代** | 432 (第 3 代) | 432 (第 3 代) | 528 (第 4 代) | 528 (第 4 代) | **N/A** <br>(16 NPU Cores) | 4090/H100 支持 **FP8** 精度训练/推理；A100 最低只支持 BF16/FP16。<br>**[5090]** 新增支持 **FP4**，为未来的极低精度推理做好了准备。<br>**[H200]** 与 H100 一样支持 FP8，但在显存受限的场景下（Memory Bound），Tensor Core 利用率更高。<br>**[A800]** Tensor Core 数量与 A100 一致，矩阵运算理论峰值未变。<br>**[M4 Max]** 依靠 Neural Engine (NPU) 加速 INT8/FP16，推理能效极高，但缺乏 Tensor Core 带来的训练加速通用性。 |
| **FP16 算力** | 165 TFLOPS | **210 TFLOPS** | 312 TFLOPS | 312 TFLOPS | 989 TFLOPS | 989 TFLOPS | ~35 TFLOPS <br>(Est. GPU) | H100 在矩阵运算上遥遥领先；但 4090 单卡算力其实略强于 A100。<br>**[5090]** 算力翻倍，理论指标已超越 A100，逼近 H100/H200。<br>**[H200]** 算力指标与 H100 一致，优势全在数据搬运速度上。<br>**[A800]** 算力未受限，主要算力指标与 A100 相同。<br>**[M4 Max]** 纯算力显著弱于 NVIDIA 旗舰卡 (约差一个数量级)，不适合从头训练模型，但在做 Batch Size=1 的推理时，算力通常不是瓶颈。 |
| **卡间互联** | 不支持 (PCIe Only) | **不支持** (PCIe Only) | **NVLink (600 GB/s)** | **NVLink (400 GB/s)** | **NVLink (900 GB/s)** | **NVLink (900 GB/s)** | **N/A** | 4090 做多卡训练效率极低 (通信瓶颈)；SXM 卡专为分布式集群设计。<br>**[5090]** 依然无 NVLink，多卡扩展性仍是其最大短板。<br>**[H200]** 维持 H100 的高速互联，完美兼容现有 H100 集群架构。<br>**[A800]** **关键阉割点**。NVLink 带宽从 600GB/s 降至 400GB/s，导致多卡大规模集群训练时通信效率下降，扩展性不如 A100。<br>**[M4 Max]** 单体强劲，但无法通过高速互联扩展，仅适合单机任务。 |
| **典型功耗** | 450W | **~600W** | 400W | 400W | 700W | 700W | **~70-100W** | H100 功耗极高，通常需要专用机房/液冷散热。<br>**[5090]** 功耗激增，对个人工作站电源和散热提出严苛要求。<br>**[H200]** 功耗维持不变，能效比（Performance per Watt）显著提升。<br>**[A800]** 功耗与 A100 相同。<br>**[M4 Max]** **能效之王**。功耗仅为 NVIDIA 旗舰卡的 1/5 到 1/10，适合本地离线、长续航的推理开发环境。 |

- A100 不支持 FP8。FP8 是从 40 系列/Hopper 架构开始引入的。这对大模型微调和量化推理是一个分水岭。
- H200 > H100 > H800 > H20
    - A800 是 A100 的阉割版，H800 是 H100 的阉割版，阉割的主要是 nvlink 的通信带宽
- 卡间通信
    - PCIe-only：GPU A $\leftrightarrow$ PCIe 插槽 $\leftrightarrow$ CPU (PCIe Root Complex/Switch) $\leftrightarrow$ System RAM $\leftrightarrow$ GPU B。
        - CPU <-> GPU: PCIe 代数/通道，CPU RAM 代数（DDRx）
    - NVLink：GPU A $\leftrightarrow$ NVLink Bridge / NVSwitch $\leftrightarrow$ GPU B。
- LLM 的训练和推理
    - 训练 (Training)： 是计算密集型 + 通信密集型任务（看重 算力 和 NVLink）。
    - 推理 (Inference)： 是显存带宽密集型任务（看重 显存带宽 和 容量）。
- pro 6000
    - 单机推理和训练
    - 4卡4090的加强版，3卡5090的加强版，不存在卡间通信的问题；

- 卡间topo：`nvidia-smi topo -m`
    - NV / NV#：表示通过 NVLink 连接（速度最快，推荐）。
    - PIX / PXB / PHB：表示通过 PCIe 总线连接（速度较慢）。
    - X: self

In [3]:
21 * 384 / 8, 5.3 * 5120 / 8, 3.2 * 5120 / 8

(1008.0, 3392.0, 2048.0)

### llm inference

> 带宽决定推理速度

$$
\text{生成速度 (Token/s)} \approx \frac{\text{显存带宽 (GB/s)}}{\text{模型大小 (GB)}}
$$
- LLM 推理是访存密集型
    - auto-regression 的时候，token by token，算力完全够用，甚至说“算力实在太快了，快到即使它闲置 99% 的时间，也就是在等数据。”
- llama-3-70b, h100
    - 显存带宽：3,350 GB/s, FP16 算力：约 1,000 TFLOPS
    - 搬运数据：把 70B 参数（140GB）从显存搬到计算核心，140GB/3,350 ~ 42ms
    - 计算：
        - $\text{计算量} = 70 \times 10^9 \text{ params} \times 2 \text{ FLOPS/param} = 140 \text{ GFLOPS} $
        - $T_{compute} = \frac{\text{计算量}}{\text{算力}} = \frac{140 \text{ GFLOPS}}{1,000,000 \text{ GFLOPS/s}} = \mathbf{0.00014 \text{ 秒}} \quad (\approx 0.14 \text{ ms})$

### 计算密集型 (Compute Bound) vs. 访存密集型 (Memory Bound)

- 计算密集型 (Compute Bound): 卷积 (Conv)、全连接层 (Linear/Dense)。这些操作数据量小，计算量大。这时算力决定速度。
    - Attention 和 FFN 
- 访存密集型 (Memory Bound): 激活函数 (ReLU/GELU)、归一化 (LayerNorm/BatchNorm)、Element-wise 操作 (Dropout, Add)、以及优化器更新 (AdamW)。这些操作计算极其简单，但需要频繁读写显存。这时带宽决定速度。
    - LayerNorm、Softmax 和 KV Cache

### 算术强度

- 算术强度 (Arithmetic Intensity)

$$
\text{算术强度} = \frac{\text{计算量 (FLOPs)}}{\text{访存量 (Bytes)}}
$$

- llm inference：Prefill vs. Decode (LLM 的推理过程并不是均质的)
    - Prefill（预填充/首词生成）： 处理你输入的 Prompt。这是一次性并行处理所有输入 Token。
        - 特性： 计算密集型 (Compute Bound)。
        - 操作： 矩阵-矩阵乘法 (GEMM)。
        - 场景：长文本输入，比如一个pdf的输入
    - Decode（解码/生成）： 一个接一个地吐出 Token。
        - 特性： 显存带宽受限 (Memory Bandwidth Bound)。
        - 操作： 矩阵-向量乘法 (GEMV)。
    - 问题的核心就在这个“Decode”阶段。
- 为什么 Decode 是带宽瓶颈？
    - 假设我们运行一个模型（比如 Qwen-7B），数据精度为 FP16（2 Bytes）。生成 1 个 Token (Batch Size = 1)
        - 它必须把模型的所有权重参数从显存（VRAM）搬运到计算核心（SRAM/Registers）。
        - 进行一次前向传播计算。
    - 搬运量和计算量
        - 搬运量 (Bytes): 假设模型参数量为 $P$，精度为 2 Bytes。你需要搬运 $2P$ Bytes 的数据。
        - 计算量 (FLOPs): 对于 Transformer 架构，生成一个 Token 的计算量大约是 $2P$ FLOPs（每个参数对应一次乘法和一次加法）。
        - 算术强度 (Arithmetic Intensity)$$\text{算术强度} = \frac{\text{计算量 (FLOPs)}}{\text{访存量 (Bytes)}} = \frac{2P}{2P} \approx 1 \text{ FLOP/Byte}$$
            - 在 Decode 阶段（Batch=1），每从显存搬运 1 个 Byte 的数据，GPU 核心只进行了 1 次浮点运算。
        - 峰值算力 (FP16/BF16): 假设约为 100 TFLOPS (100,000 GFLOPS)。显存带宽: 约为 1,800 GB/s。
            - $\text{硬件理想比率} = \frac{100,000 \text{ GFLOPS}}{1,800 \text{ GB/s}} \approx 55 \text{ FLOPs/Byte}$
        - 算力被浪费掉了

- RTX 4090 生成 1 个 Token 的时间线
    - 搬运时间 (IO Time) —— “传送带速度”，$\text{搬运时间} = \frac{\text{模型大小}}{\text{显存带宽}} = \frac{14 \text{ GB}}{1,008 \text{ GB/s}} \approx 0.0138 \text{ 秒} \approx \mathbf{13.9 \text{ ms}}$
    - 计算时间 (Compute Time) —— “切菜速度”, $\text{计算时间} = \frac{\text{计算量}}{\text{算力}} = \frac{14 \text{ GFLOPs}}{83,000 \text{ GFLOPs/s}} \approx 0.00017 \text{ 秒} \approx \mathbf{0.17 \text{ ms}}$
        - fp32：83 TFLOPS，fp16：330 TFLOPS